<a href="https://colab.research.google.com/github/niksisons/neural_networks/blob/main/10.%20%D0%98%D1%81%D0%BF%D0%BE%D0%BB%D1%8C%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5%20%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8%20YOLOv9%20%D0%B4%D0%BB%D1%8F%20%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D0%B5%20%D0%B7%D0%B0%D0%B4%D0%B0%D1%87%20%D0%B4%D0%B5%D1%82%D0%B5%D0%BA%D1%86%D0%B8%D0%B8%20%D0%B8%20%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%969_%D0%98%D1%81%D0%BF%D0%BE%D0%BB%D1%8C%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8_YOLOv12_%D0%B4%D0%BB%D1%8F_%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D1%87_%D0%B4%D0%B5%D1%82%D0%B5%D0%BA%D1%86%D0%B8%D0%B8_%D0%B8_%D0%BA%D0%BB%D0%B0%D1%81%D1%81%D0%B8%D1%84%D0%B8%D0%BA%D0%B0%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №9. Использование модели YOLOv26 для решение задач детекции и классификации**

## **Задание 1. Знакомство с новейшей версией модели YOLO**

- Ознакомьтесь с технической документацией по новейшей версии модели YOLOv26: https://docs.ultralytics.com/ru/models/yolo26/

- Рассмотрите пример обучения этой модели c использованием датасетов, созданных в Roboflow: https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolo26-object-detection-on-custom-dataset.ipynb?ref=blog.roboflow.com

### **Ответьте на вопрос: Какие принципиальные отличия появились в модели YOLOv26 относительно предыдущих версий?**

**Принципиальные отличия YOLOv26 от предыдущих версий:**

1. **Сквозной вывод без NMS (End-to-End):** Как и в последних версиях (с v10), YOLOv26 работает без этапа постобработки NMS, генерируя прогнозы напрямую. Это снижает задержку и делает вывод в реальном времени быстрее и стабильнее.
2. **Удаление DFL:** Модуль Distribution Focal Loss (DFL) был удален для упрощения инференса и расширения поддержки периферийных Edge и IoT устройств с низким энергопотреблением.
3. **Новый оптимизатор MuSGD:** Представлен гибридный оптимизатор (SGD + Muon), заимствованный из подходов обучения LLM (Kimi K2). Он обеспечивает более стабильное обучение и быструю сходимость.
4. **Улучшение потерь (ProgLoss + STAL):** Новые функции лосса колоссально улучшили распознавание *мелких объектов*, что критически важно в аэрофотосъемке.
5. **Оптимизация производительности:** Вывод на CPU стал до 43% быстрее за счет оптимизации архитектуры под Edge-вычисления.
6. **Улучшения в смежных задачах:** Интеграция RLE для более точной оценки поз (Pose Estimation), введение потерь семантической сегментации для повышения качества масок и специализированные функции потерь для OBB (Oriented Bounding Boxes).

*Примечание:* для каждого последующего задания, в конечном итоге, необходимо сформировать обученную модель и задеплоить её на сервисе **RoboFlow**



## **Задание 2. Найдите готовый датасет для детекции объектов и обучите на нем модель YOLOv26:**

- Для удобства работы с датасетом и деплоя модели используйте сервис [RoboFlow](https://roboflow.com/)

In [ ]:
# 1. Установка необходимых библиотек
!pip install -q ultralytics roboflow

import locale
# Фикс для кодировки в Colab
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="fhRkq4UGLeKk6BeMAG8h")
project = rf.workspace("roboflow-58fyf").project("rock-paper-scissors-sxsw")
version = project.version(14)
dataset = version.download("yolo26")
                

In [ ]:
from ultralytics import YOLO

# 1. Инициализация модели YOLOv26
model = YOLO("yolo26n.pt") # Загружаем предобученную модель YOLOv26 nano

# 2. Обучение модели на встроенном тестовом датасете COCO8.
# Указав data="coco8.yaml", библиотека ultralytics АВТОМАТИЧЕСКИ 
# скачает этот готовый датасет (мини-версия COCO на 8 изображений для тестов) 
# и начнет на нём обучение. Никакие API ключи и регистрации не нужны!
results = model.train(data="coco128.yaml", epochs=10, imgsz=640)

# 3. Вывод метрик валидации
metrics = model.val()
print("Mean Average Precision (mAP50-95):", metrics.box.map)

# Пример инференса на дефолтном изображении (удобно для проверки детекции)
# model.predict(source="https://ultralytics.com/images/bus.jpg", save=True, imgsz=640)

## **Задание 3. Сформируйте свой датасет для детекции объектов и обучите на нем модель YOLOv26**



Требования к датасету:

- Количество изображений в датасете: минимум 90
- Количество классов: более 2х
- Обязательно должны присутствовать изображения, содержащие несколько классов одновременно


*Примечание: Формирование датасета включает в себя поиск изображений и ручное аннотирование объектов на изображениях.*

In [ ]:
# Важно: Чтобы не перезатирать старые данные, используем новый проект.
# Вставьте сюда свежий код из Roboflow, который вы сгенерировали для своей КАСТОМНОЙ разметки (минимум 3 класса, 90 изображений)


rf = Roboflow(api_key="fhRkq4UGLeKk6BeMAG8h")
project = rf.workspace("systemmonitoring").project("task9-tslhp")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
# Инициализация новой модели YOLO
# Если хотите дообучить (Transfer Learning) предыдущую модель, подставьте путь к её весам:
# custom_model = YOLO("runs/detect/train/weights/best.pt") 
custom_model = YOLO("yolo26n.pt") # Снова используем предобученные веса YOLOv26 nano

# Обучаем модель на собственном наборе данных (data.yaml)
# Убедитесь, что epochs и imgsz подходят для ваших целей, 10-15 эпох для тестов вполне достаточно
results_custom = custom_model.train(data=f"{custom_dataset.location}/data.yaml", epochs=20, imgsz=640)

# Вывод метрик валидации
metrics_custom = custom_model.val()
print("Метрики вашего кастомного датасета:")
print("mAP50-95:", metrics_custom.box.map)

## **Задание 4 (для магистрантов). Обучите модель YOLOv26 на датасете [RSD-GOD](https://github.com/Dr-Zhuang/geospatial-object-detection). Затем самостоятельно сформируйте тестовые данные (не менее 50 изображений с аннотациями) и оцените качество обученной модели:**



**Прямая ссылка на загрузку датасета: [загрузить](https://drive.google.com/open?id=1ttvSta0BRxW7tTV_st89vSb_obHVre34)**

**Ссылка на датасет в среде roboflow universe:** https://universe.roboflow.com/animals-sqrdn/rsd-god

 *Примечание: для поиска данных удобно использовать сервис Google Earth/Google Earth Engine*

In [ ]:
# Ваш код